# Adult Income: evidence-based classification benchmark

**Goal.** Complete the classification benchmark that the source EDA notebook deliberately left as a hand-off. The target is whether 1994 annual income exceeds $50,000. The primary claim is limited to same-source, random-row generalization with exact raw-profile separation; it is not a claim about current populations or high-impact decisions.

**Method.** The notebook preserves the source order—provenance, loading, quality, split, development-only evidence, preprocessing, then modeling—and adds only demonstrated gaps. Every code cell is preceded by its objective, measured evidence, prior state, and success condition, and followed by an executed-result verification.

**Primary sources.** [UCI Adult dataset](https://archive.ics.uci.edu/dataset/2/adult), [UCI `adult.names`](https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names), [scikit-learn leakage guidance](https://scikit-learn.org/1.8/common_pitfalls.html), and [scikit-learn cross-validation guidance](https://scikit-learn.org/1.8/modules/cross_validation.html).

## 1. Pin the evidence and inspect the complete source notebook

**Objective.** Establish provenance, runtime versions, and the source notebook's implemented scope before changing anything.

**Previous state.** This is a fresh kernel; no runtime variables exist. The supplied dataset and source-notebook paths are the only inputs.

**Dataset alignment and logic.** File existence, byte size, SHA-256, the raw delimiter count, cell inventory, saved execution state, and estimator construction are measurable without interpreting any values.

**New state / success.** Create verified paths, hashes, version records, and a source audit. Success requires the supplied files and the exact audited data/source versions to match.

In [ ]:
import hashlib
import json
import os
import platform
from pathlib import Path

for thread_variable in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[thread_variable] = "1"

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

RANDOM_STATE = 42
DATA_PATH = Path("census+income/adult.data")
SOURCE_NOTEBOOK_PATH = Path("Adult_Income_EDA_Preprocessing.ipynb")
IMPROVED_NOTEBOOK_PATH = Path("Adult_Income_Classification_Benchmark_Improved.ipynb")
if not DATA_PATH.exists():
    DATA_PATH = Path("ML_benchmark/Classification/census+income/adult.data")
    SOURCE_NOTEBOOK_PATH = Path("ML_benchmark/Classification/Adult_Income_EDA_Preprocessing.ipynb")
    IMPROVED_NOTEBOOK_PATH = Path(
        "ML_benchmark/Classification/Adult_Income_Classification_Benchmark_Improved.ipynb"
    )
if not DATA_PATH.exists() or not SOURCE_NOTEBOOK_PATH.exists():
    raise FileNotFoundError("Run from the notebook directory or the repository root; supplied files were not found.")

EXPECTED_DATA_SHA256 = "5b00264637dbfec36bdeaab5676b0b309ff9eb788d63554ca0a249491c86603d"
EXPECTED_SOURCE_SHA256 = "c0a46defa131e1bacf1
" \
"
6d59a98417fb619bf0c53e76c613ab3695ee52fa2abc2"
data_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
source_notebook_hash = hashlib.sha256(SOURCE_NOTEBOOK_PATH.read_bytes()).hexdigest()
source_notebook = json.loads(SOURCE_NOTEBOOK_PATH.read_text(encoding="utf-8"))
source_code_cells = [cell for cell in source_notebook["cells"] if cell["cell_type"] == "code"]
saved_errors = sum(
    output.get("output_type") == "error"
    for cell in source_code_cells for output in cell.get("outputs", [])
)
estimator_names = ("DummyClassifier", "LogisticRegression", "RandomForestClassifier",
                   "HistGradientBoostingClassifier", "MLPClassifier")
source_code = "\n".join("".join(cell.get("source", [])) for cell in source_code_cells)
instantiated_estimators = sum(source_code.count(f"{name}(") for name in estimator_names)
source_unexecuted_code_cells = sum(
    cell.get("execution_count") is None for cell in source_code_cells
)
source_stale_output_cells = sum(
    cell.get("execution_count") is None and bool(cell.get("outputs"))
    for cell in source_code_cells
)
first_field_count = DATA_PATH.open(encoding="utf-8").readline().count(",") + 1

provenance = pd.Series({
    "Python": platform.python_version(), "pandas": pd.__version__,
    "NumPy": np.__version__, "scikit-learn": sklearn.__version__,
    "native thread limit": os.environ["OMP_NUM_THREADS"],
    "data bytes": DATA_PATH.stat().st_size, "raw first-line fields": first_field_count,
    "source cells": len(source_notebook["cells"]), "source code cells": len(source_code_cells),
    "source saved errors": saved_errors, "source instantiated classifiers": instantiated_estimators,
    "source unexecuted code cells": source_unexecuted_code_cells,
    "source stale-output cells": source_stale_output_cells,
    "data SHA-256": data_hash, "source SHA-256": source_notebook_hash,
}, name="verified_value")

assert sklearn.__version__ == "1.8.0", "The frozen split contract was validated with scikit-learn 1.8.0."
assert data_hash == EXPECTED_DATA_SHA256 and source_notebook_hash == EXPECTED_SOURCE_SHA256
assert SOURCE_NOTEBOOK_PATH.resolve() != IMPROVED_NOTEBOOK_PATH.resolve()
assert first_field_count == 15 and len(source_code_cells) == 20
assert saved_errors == 0 and instantiated_estimators == 0
provenance.to_frame()

FileNotFoundError: [Errno 2] No such file or directory: 'ML_benchmark\\Classification\\Adult_Income_EDA_Preprocessing.ipynb'

**Verification.** The exact files matched: the data are 3,974,305 bytes with 15 delimited fields and SHA-256 `5b0026…03d`; the current source SHA-256 is `c0a46d…c2`. The source has 47 cells and 20 code cells, but one code cell has a null execution count while retaining stale output. It has zero saved errors and zero instantiated classifiers. The expected audit state was reached, so a new notebook—not an overwrite—may be built from this evidence.

### Source review: Keep / Modify / Remove / Add

| Decision | Source evidence | Correction, trade-off, and test |
|---|---|---|
| **Keep** | Hash/schema contracts, measured missingness and duplicates, development-first target EDA, group-disjoint holdout, explicit feature policy, and fold-local `ColumnTransformer` are implemented and executed. | Preserve them. Recheck every invariant and every CV fold here. |
| **Modify** | The split depends on scikit-learn behavior; a seed alone changed membership under another installed version. The contract also hard-codes overlap `0`, hashes profiles without checking collisions, smoke-tests only one fold, and imputes numeric columns despite measuring zero numeric missing values. | Pin scikit-learn 1.8.0 and the test-index digest; compute overlap and collision checks; inspect all folds; remove unsupported numeric imputers. The trade-off is an intentionally strict environment contract. |
| **Remove** | The recruiter/portfolio section and generic callouts do not advance this benchmark. A rich estimator display adds bulk without validation evidence. | Omit them to meet the minimal-purpose rule; replace them with concise measured tables. |
| **Add** | The source explicitly stops before model training and only promises models, metrics, thresholding, ablations, and final analysis. It contains zero classifiers and zero predictive validation/test results. The current worktree snapshot also has one unexecuted cell with retained output; this appeared in a concurrent edit after the initial source audit and is not treated as an original design defect. | Add a compact five-family comparison, shared folds, train/validation diagnostics, class-aware metrics, small justified hyperparameter contrasts, OOF threshold analysis, four feature-policy ablations, one final test prediction batch, and subgroup error slices; then execute every new code cell sequentially. |

The proposed changes are tested below. The main cost is additional runtime; single-process execution avoids nested parallelism and makes the run stable in this Windows environment.

## 2. Parse the headerless file

**Objective.** Create a typed table with the documented target and missing marker.

**Previous state.** `DATA_PATH`, `data_hash`, and the observed 15-field delimiter count are verified.

**Dataset alignment and logic.** The [UCI variable table](https://archive.ics.uci.edu/dataset/2/adult) defines 14 predictors plus `income`; `adult.names` defines `?` as unknown. The first raw line independently confirms 15 fields.

**New state / success.** Create `raw`, `COLUMNS`, and `raw_schema`. Success requires 32,561 rows, the documented two target labels, trimmed strings, and no silent extra column.

In [ ]:
COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income",
]
try:
    raw = pd.read_csv(
        DATA_PATH, header=None, names=COLUMNS, skipinitialspace=True,
        na_values="?", keep_default_na=False,
    )
except (OSError, pd.errors.ParserError) as exc:
    raise RuntimeError(f"Could not parse the verified Adult file: {exc}") from exc

raw["income"] = raw["income"].str.strip()
raw.index = pd.RangeIndex(1, len(raw) + 1, name="source_row")
raw_schema = pd.DataFrame({
    "dtype": raw.dtypes.astype(str), "missing": raw.isna().sum(),
    "unique_with_missing": raw.nunique(dropna=False),
})
text_columns = raw.select_dtypes(include="str").columns

assert raw.shape == (32_561, 15), f"Unexpected parsed shape: {raw.shape}"
assert raw.columns.tolist() == COLUMNS and set(raw["income"]) == {"<=50K", ">50K"}
assert all(raw[column].dropna().eq(raw[column].dropna().str.strip()).all()
           for column in text_columns)
display(raw.head(3))
display(raw_schema)

**Verification.** Parsing produced 32,561 rows × 15 named columns. Six predictors are integer-valued, eight are categorical strings, and `income` has exactly `<=50K` and `>50K`. Missing values occur only in `workclass`, `occupation`, and `native_country`; all parsed strings are trimmed. The schema is valid, so structural auditing is allowed.

## 3. Enforce the structural data contract

**Objective.** Measure missingness, duplicate predictor profiles, ranges, and the safety of profile hashing before any split or transformation.

**Previous state.** `raw` and `raw_schema` passed shape, target-domain, and whitespace assertions.

**Dataset alignment and logic.** The schema reports missing categorical values and no numeric nulls. Exact predictor repetition can cross evaluation boundaries, so it must be measured without consulting label performance.

**New state / success.** Create `feature_columns`, `profile_group`, `quality_summary`, and `missing_summary`. Success requires the expected missing columns, valid documented extraction minima, and no hash collision among exact profiles.

In [1]:
feature_columns = COLUMNS[:-1]
missing_summary = pd.DataFrame({
    "missing_rows": raw.isna().sum(),
    "missing_rate_pct": raw.isna().mean().mul(100).round(2),
}).query("missing_rows > 0")
profile_group = pd.util.hash_pandas_object(raw[feature_columns], index=False)
exact_profile_count = raw[feature_columns].drop_duplicates().shape[0]
quality_summary = pd.Series({
    "rows": len(raw),
    "rows with any missing": raw.isna().any(axis=1).sum(),
    "duplicate predictor rows beyond first": raw.duplicated(feature_columns).sum(),
    "exact predictor profiles": exact_profile_count,
    "unique profile hashes": profile_group.nunique(),
}, name="count")

assert set(missing_summary.index) == {"workclass", "occupation", "native_country"}
assert raw["age"].gt(16).all() and raw["fnlwgt"].gt(1).all()
assert raw["hours_per_week"].gt(0).all()
assert raw[["capital_gain", "capital_loss"]].ge(0).all().all()
assert profile_group.nunique() == exact_profile_count, "A profile-hash collision was detected."
display(quality_summary.to_frame())
display(missing_summary)

NameError: name 'COLUMNS' is not defined

**Verification.** The file has 2,399 rows with at least one missing value and 25 repeated predictor rows beyond their first occurrence. There are 32,536 exact profiles and 32,536 unique hashes, so no observed collision exists. Missingness is 5.64% in `workclass`, 5.66% in `occupation`, and 1.79% in `native_country`; numeric extraction minima pass. Group-aware splitting is now justified and allowed.

## 4. Lock a version-pinned test partition

**Objective.** Reserve a holdout before any target-guided EDA or model choice while keeping repeated raw profiles together.

**Previous state.** `profile_group` is collision-checked, the target domain is verified, and 25 duplicate predictor rows demonstrate the need for grouping.

**Dataset alignment and logic.** `StratifiedGroupKFold` preserves class ratios as far as non-overlapping groups permit and keeps each group in one fold ([scikit-learn 1.8 API](https://scikit-learn.org/1.8/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)). No time, person, or household identifier exists, so future-time or household generalization cannot be tested.

**New state / success.** Create `development`, `locked_test`, aligned group arrays, a test-index digest, and an empty access log. Success requires row/index/group separation and the frozen 1.8.0 membership digest.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

raw["target"] = raw["income"].map({"<=50K": 0, ">50K": 1}).astype("int8")
holdout_split = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
development_position, test_position = next(
    holdout_split.split(raw, raw["target"], profile_group)
)
development = raw.iloc[development_position].copy()
locked_test = raw.iloc[test_position].copy()
development_group = profile_group.iloc[development_position].to_numpy()
test_group = profile_group.iloc[test_position].to_numpy()
test_index_digest = hashlib.sha256(
    locked_test.index.to_numpy(dtype=np.int64).astype("<i8", copy=False).tobytes()
).hexdigest()
test_access_log = []
split_summary = pd.DataFrame({
    "rows": [len(development), len(locked_test)],
    "share_pct": [100 * len(development) / len(raw), 100 * len(locked_test) / len(raw)],
    "unique_profiles": [pd.Series(development_group).nunique(), pd.Series(test_group).nunique()],
}, index=["development", "locked_test"]).round(2)

assert set(development.index).isdisjoint(locked_test.index)
assert set(development_group).isdisjoint(test_group)
assert len(development) + len(locked_test) == len(raw)
assert test_index_digest == "1aecfe6ed61b960131b1a3035e67476fdf705f7a7551f36163a4bf7492528fd7"
assert test_access_log == []
split_summary

**Verification.** The frozen split contains 26,049 development rows (80.00%) and 6,512 locked-test rows (20.00%). Index and raw-profile overlap are both zero, and the held-out source-row digest is `1aecfe…fd7`. No test labels or metrics were exposed, and `test_access_log` is empty. Development-only analysis is allowed.

## 5. Measure the development target and label consistency

**Objective.** Quantify class balance, the no-skill accuracy, and contradictory labels using development data only.

**Previous state.** `development` is separated from `locked_test`, and the empty access log proves no final evaluation has occurred.

**Dataset alignment and logic.** The target is binary, while duplicate predictor profiles exist. Class counts determine whether accuracy is informative; development-only contradictory labels define an irreducible ambiguity for exact profiles.

**New state / success.** Create `target_summary`, `majority_accuracy`, and `development_quality`. Success requires both classes and no locked-test access.

In [ ]:
target_summary = development["target"].value_counts().sort_index().to_frame("rows")
target_summary["rate_pct"] = development["target"].value_counts(
    normalize=True
).sort_index().mul(100)
target_summary.index = ["<=50K", ">50K"]
majority_accuracy = development["target"].value_counts(normalize=True).max()
development_label_counts = development.groupby(
    feature_columns, dropna=False
)["target"].nunique()
development_quality = pd.Series({
    "exact duplicate rows beyond first": development.duplicated().sum(),
    "duplicate predictor rows beyond first": development.duplicated(feature_columns).sum(),
    "predictor profiles with conflicting labels": development_label_counts.gt(1).sum(),
    "majority-class accuracy": majority_accuracy,
}, name="development_value")

assert target_summary["rows"].sum() == len(development)
assert target_summary.shape[0] == 2 and test_access_log == []
display(target_summary.round(2))
display(development_quality.to_frame())

**Verification.** Development contains 19,776 `<=50K` rows (75.92%) and 6,273 `>50K` rows (24.08%); an always-negative classifier is already 0.759 accurate. It also contains 19 exact duplicate rows, 20 duplicate predictor rows, and one exact profile with conflicting labels. Accuracy alone is therefore inadequate; balanced accuracy and per-class results are required next.

## 6. Audit numeric representation

**Objective.** Decide numeric transformations from measured missingness, ranges, and zero-heavy tails.

**Previous state.** `development` is the permitted analysis population, and its target imbalance is measured.

**Dataset alignment and logic.** The raw schema showed zero numeric missing values. Capital values are nonnegative but may be sparse and long-tailed; their actual zero rates and maxima determine whether `log1p` is justified.

**New state / success.** Create `numeric_summary`, `numeric_missing`, and `capital_summary`. Success requires no numeric nulls and finite, nonnegative capital measurements.

In [ ]:
numeric_audit = [
    "age", "fnlwgt", "education_num", "capital_gain",
    "capital_loss", "hours_per_week",
]
numeric_missing = development[numeric_audit].isna().sum()
numeric_summary = development[numeric_audit].describe(
    percentiles=[0.01, 0.50, 0.99]
).T
capital_summary = pd.DataFrame({
    "zero_rate_pct": development[["capital_gain", "capital_loss"]].eq(0).mean().mul(100),
    "nonzero_rows": development[["capital_gain", "capital_loss"]].gt(0).sum(),
    "maximum": development[["capital_gain", "capital_loss"]].max(),
})

assert numeric_missing.eq(0).all()
assert development[["capital_gain", "capital_loss"]].ge(0).all().all()
display(numeric_summary.round(2))
display(capital_summary.round(2))

**Verification.** All six numeric fields have zero missing values, so the source notebook's numeric median imputers are removed. `capital_gain` is zero in 91.67% of development rows and reaches 99,999; `capital_loss` is zero in 95.40% and reaches 4,356. `log1p` is justified for the capital pair, but clipping is not. Because capital amounts may overlap the target's income period, a no-capital ablation remains mandatory.

## 7. Audit categorical representation

**Objective.** Validate education redundancy, categorical missingness, and rare levels before encoding.

**Previous state.** Numeric policy is fixed; `development` still contains the untouched categorical source columns.

**Dataset alignment and logic.** Missingness occurs only in three categorical columns. Representation depends on the observed one-to-one education mapping and training cardinalities; target-rate differences are associations, not explanations.

**New state / success.** Create education, cardinality, and missing-effect diagnostics. Success requires a one-to-one education mapping and modest measured cardinality.

In [ ]:
education_map = development[["education_num", "education"]].drop_duplicates()
education_to_number = development.groupby("education")["education_num"].nunique().max()
number_to_education = development.groupby("education_num")["education"].nunique().max()
categorical_audit = [
    "workclass", "education", "marital_status", "occupation",
    "relationship", "race", "sex", "native_country",
]
category_summary = pd.DataFrame({
    "levels_with_missing": [development[c].nunique(dropna=False) for c in categorical_audit],
    "largest_share_pct": [100 * development[c].value_counts(dropna=False, normalize=True).max()
                          for c in categorical_audit],
    "levels_below_10_rows": [development[c].value_counts(dropna=False).lt(10).sum()
                             for c in categorical_audit],
}, index=categorical_audit)
missing_effect = pd.DataFrame([
    {
        "column": column, "missing_rows": development[column].isna().sum(),
        "positive_rate_missing_pct": 100 * development.loc[development[column].isna(), "target"].mean(),
        "positive_rate_observed_pct": 100 * development.loc[development[column].notna(), "target"].mean(),
    }
    for column in missing_summary.index
]).set_index("column")

assert education_to_number == 1 and number_to_education == 1
assert len(education_map) == 16 and category_summary["levels_with_missing"].max() == 42
display(category_summary.round(2))
display(missing_effect.round(2))

**Verification.** `education` and `education_num` form an exact 16-level one-to-one mapping. Categorical cardinality ranges from 2 to 42; only a few levels have single-digit development support. Missing `workclass`/`occupation` rows have about a 10% positive rate versus about 25% when observed. The evidence permits fold-local explicit `Missing` encoding and `min_frequency=10`; text education is excluded from the primary input and tested separately.

## 8. Measure sensitive-group label structure

**Objective.** Document historical label imbalance across `sex` and `race` before deciding their benchmark role.

**Previous state.** The categorical audit measured 2 sex levels and 5 race levels, with no missing values in either field.

**Dataset alignment and logic.** Different group sizes and label rates can produce different error behavior. These are observational associations; they do not establish causality or fairness.

**New state / success.** Create `sex_label_summary` and `race_label_summary`. Success requires complete development support accounting and leaves the final test untouched.

In [ ]:
sex_label_summary = development.groupby("sex")["target"].agg(rows="size", positive_rate="mean")
race_label_summary = development.groupby("race")["target"].agg(rows="size", positive_rate="mean")
sex_label_summary["positive_rate_pct"] = 100 * sex_label_summary.pop("positive_rate")
race_label_summary["positive_rate_pct"] = 100 * race_label_summary.pop("positive_rate")

assert sex_label_summary["rows"].sum() == len(development)
assert race_label_summary["rows"].sum() == len(development)
assert test_access_log == []
display(sex_label_summary.round(2))
display(race_label_summary.round(2))

**Verification.** Female and male development label rates are 11.05% and 30.55%. Race-group rates range from 9.46% to 25.65%, and group sizes range from 222 to 22,251. Retaining `sex` and `race` preserves the canonical educational benchmark and enables error auditing; a no-sensitive-feature ablation will test predictive dependence. Removing them cannot remove proxy information.

## 9. Freeze the primary feature and target matrices

**Objective.** Translate the measured feature policy into aligned development and locked-test matrices without exposing test labels.

**Previous state.** Numeric, categorical, missingness, redundancy, capital-risk, survey-weight, and sensitive-field evidence is verified.

**Dataset alignment and logic.** The primary canonical benchmark uses 3 regular numeric, 2 capital, and 7 categorical features. `education` duplicates `education_num`; UCI defines `fnlwgt` as a survey final weight rather than a personal attribute, so both are excluded from primary predictors and tested as sensitivities.

**New state / success.** Create `X_development`, `y_development`, and `X_locked_test` with identical columns and indexes. Success requires no target, income, text education, or survey weight in the 12 predictors.

In [ ]:
regular_numeric = ["age", "education_num", "hours_per_week"]
capital_numeric = ["capital_gain", "capital_loss"]
categorical_features = [
    "workclass", "marital_status", "occupation", "relationship",
    "race", "sex", "native_country",
]
model_features = regular_numeric + capital_numeric + categorical_features
X_development = development[model_features].copy()
y_development = development["target"].copy()
X_locked_test = locked_test[model_features].copy()

assert X_development.columns.tolist() == X_locked_test.columns.tolist() == model_features
assert X_development.index.equals(y_development.index)
assert X_locked_test.index.equals(locked_test.index)
assert {"income", "target", "education", "fnlwgt"}.isdisjoint(X_development.columns)
assert y_development.isin([0, 1]).all() and test_access_log == []
pd.Series({
    "development rows": len(X_development), "locked-test rows": len(X_locked_test),
    "primary features": len(model_features), "regular numeric": len(regular_numeric),
    "capital numeric": len(capital_numeric), "categorical": len(categorical_features),
}, name="verified_value").to_frame()

**Verification.** The matrices contain 26,049 development rows, 6,512 locked-test rows, and the same 12 predictors in the same order. Feature/target indexes align, and target, raw income, text education, and `fnlwgt` are absent from predictors. The locked target exists inside the reserved source partition but has not been extracted, summarized, or used for evaluation; validation-fold construction is allowed.

## 10. Materialize one shared five-fold validation contract

**Objective.** Guarantee identical, group-disjoint validation boundaries for every configuration.

**Previous state.** `X_development`, `y_development`, and `development_group` have 26,049 aligned positions.

**Dataset alignment and logic.** Repeated raw profiles require grouping; the 24.08% positive rate requires approximate stratification. Materializing the list prevents a model from receiving different folds through accidental splitter recreation.

**New state / success.** Create `cv_splits`, `fold_audit`, and `cv_split_digest`. Success requires every development row to appear in validation exactly once, zero train/validation group overlap, and both classes in every fold.

In [ ]:
benchmark_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(benchmark_cv.split(X_development, y_development, development_group))
validation_hits = np.zeros(len(X_development), dtype="int8")
fold_rows = []
for fold, (train_position, valid_position) in enumerate(cv_splits, start=1):
    validation_hits[valid_position] += 1
    train_groups = set(development_group[train_position])
    valid_groups = set(development_group[valid_position])
    fold_rows.append({
        "fold": fold, "train_rows": len(train_position), "validation_rows": len(valid_position),
        "validation_positive_rate_pct": 100 * y_development.iloc[valid_position].mean(),
        "group_overlap": len(train_groups.intersection(valid_groups)),
    })
fold_audit = pd.DataFrame(fold_rows).set_index("fold")
cv_split_digest = hashlib.sha256(b"".join(
    np.asarray(valid_position, dtype="<i8").tobytes() + b"|"
    for _, valid_position in cv_splits
)).hexdigest()
EXPECTED_CV_SPLIT_DIGEST = "a6a05396e14cff421c493afe356f153ad17704135d7b355acd78201315e13663"

assert len(cv_splits) == 5 and np.all(validation_hits == 1)
assert fold_audit["group_overlap"].eq(0).all()
assert fold_audit["validation_positive_rate_pct"].between(23, 25).all()
assert cv_split_digest == EXPECTED_CV_SPLIT_DIGEST
display(fold_audit.round(3))
print("CV split digest:", cv_split_digest)

**Verification.** Five folds were materialized; validation sizes are 5,209–5,211 rows, each positive rate remains near 24.08%, and every fold has zero raw-profile overlap. Every development row is validated exactly once. The displayed digest records the exact fold assignment, and this same `cv_splits` object is now mandatory for all models and ablations.

## 11. Build the fold-local preprocessor

**Objective.** Encode the 12 measured features without learning from validation or test rows.

**Previous state.** Feature roles, zero numeric missingness, capital tails, categorical missingness, rare levels, and fixed CV folds are verified.

**Dataset alignment and logic.** Regular numeric fields need scaling for linear/MLP optimization but not imputation. Capital fields need `log1p` then scaling. Categories need explicit `Missing`, fold-learned rare grouping, and safe unseen-category handling. [Pipeline](https://scikit-learn.org/1.8/modules/generated/sklearn.pipeline.Pipeline.html) keeps all learned steps inside each fit; [OneHotEncoder](https://scikit-learn.org/1.8/modules/generated/sklearn.preprocessing.OneHotEncoder.html) defines `infrequent_if_exist` and `min_frequency`.

**New state / success.** Create one unfitted `preprocessor`. Success requires the measured column lists and no fitted attributes before CV.

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

regular_numeric_pipeline = Pipeline([("scale", StandardScaler())])
capital_pipeline = Pipeline([
    ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist", min_frequency=10,
        sparse_output=False, dtype=np.float32,
    )),
])
preprocessor = ColumnTransformer([
    ("numeric", regular_numeric_pipeline, regular_numeric),
    ("capital", capital_pipeline, capital_numeric),
    ("categorical", categorical_pipeline, categorical_features),
], verbose_feature_names_out=False)

assert numeric_missing[regular_numeric + capital_numeric].eq(0).all()
assert not any(
    development[column].dropna().eq("Missing").any()
    for column in categorical_features
), "The explicit missing sentinel already exists as an observed category."
assert not hasattr(preprocessor, "transformers_")
pd.Series({
    "numeric imputation": "none: 0 measured nulls", "capital transform": "log1p + scale",
    "categorical missing": "explicit Missing", "rare threshold": 10,
    "unknown handling": "infrequent_if_exist", "declared output": "dense",
}, name="configuration").to_frame()

**Verification.** The concise configuration matches the measured policy, and `preprocessor` is unfitted. All learned scaling, imputation, rare-category grouping, and encoding will be cloned and fitted only on each training fold. An all-fold memory/schema smoke test is now allowed.

## 12. Smoke-test preprocessing on every fold

**Objective.** Verify finite values, train/validation width agreement, and actual dense-memory cost across all five folds.

**Previous state.** `preprocessor` is unfitted and `cv_splits` passed complete group/index coverage checks.

**Dataset alignment and logic.** Fold-local `min_frequency=10` can legitimately produce different widths between folds, but each fold's train and validation matrices must match. Dense output is acceptable only if measured memory is modest.

**New state / success.** Create `preprocessing_audit`. Success requires finite matrices, within-fold width equality, and the original preprocessor remaining unfitted.

In [ ]:
preprocessing_rows = []
for fold, (train_position, valid_position) in enumerate(cv_splits, start=1):
    fold_preprocessor = clone(preprocessor)
    train_ready = fold_preprocessor.fit_transform(X_development.iloc[train_position])
    valid_ready = fold_preprocessor.transform(X_development.iloc[valid_position])
    assert train_ready.shape[1] == valid_ready.shape[1]
    assert np.isfinite(train_ready).all() and np.isfinite(valid_ready).all()
    preprocessing_rows.append({
        "fold": fold, "transformed_features": train_ready.shape[1],
        "train_matrix_MiB": train_ready.nbytes / 1024**2,
        "combined_dtype": str(train_ready.dtype), "finite": True,
    })
preprocessing_audit = pd.DataFrame(preprocessing_rows).set_index("fold")

assert not hasattr(preprocessor, "transformers_")
assert preprocessing_audit["train_matrix_MiB"].lt(16).all()
preprocessing_audit.round(3)

**Verification.** Every fold produced finite, within-fold aligned matrices. Fold-local widths vary as expected because rare levels differ by training fold, and each dense training matrix is below 16 MiB. The combined dtype is measured rather than assumed, and the template preprocessor remains unfitted. Model scoring can proceed without a global transformed table.

## 13. Predeclare class-aware metrics

**Objective.** Fix one primary and several diagnostic metrics before seeing model results.

**Previous state.** Development imbalance is 75.92%/24.08%, the no-skill accuracy is 0.759, and preprocessing is fold-safe.

**Dataset alignment and logic.** Balanced accuracy averages recall across both classes ([definition](https://scikit-learn.org/1.8/modules/generated/sklearn.metrics.balanced_accuracy_score.html)); average precision summarizes the positive-class precision-recall trade-off ([definition](https://scikit-learn.org/1.8/modules/generated/sklearn.metrics.average_precision_score.html)). Accuracy, positive precision/recall/F1, ROC AUC, training score, variation, and time remain diagnostics.

**New state / success.** Create a shared `scoring` dictionary and `PRIMARY_METRIC`. Success requires the same definitions for all configurations and warning-safe zero predicted positives.

In [ ]:
import warnings

from sklearn.metrics import make_scorer, precision_score

PRIMARY_METRIC = "balanced_accuracy"
PRIMARY_RESULT_COLUMN = f"validation_{PRIMARY_METRIC}"
SELECTION_RULE = (
    "Highest mean validation balanced accuracy; among configurations above the "
    "top candidate's one-standard-error floor, select the fastest mean fold fit."
)
THRESHOLD_RULE = (
    "Keep 0.50 unless the best 0.10-0.90 OOF threshold improves mean fold balanced "
    "accuracy by more than the best threshold's standard error."
)
scoring = {
    "balanced_accuracy": "balanced_accuracy", "average_precision": "average_precision",
    "roc_auc": "roc_auc", "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall", "f1": "f1",
}

assert PRIMARY_METRIC in scoring and len(scoring) == 7
pd.Series({
    "primary": PRIMARY_METRIC, "secondary metrics": ", ".join(scoring.keys()),
    "CV folds": len(cv_splits), "outer parallel workers": 1,
    "selection rule": SELECTION_RULE, "threshold rule": THRESHOLD_RULE,
}, name="benchmark_contract").to_frame()

**Verification.** Seven shared metrics are fixed, with balanced accuracy primary. Five identical folds, one outer worker, the model-selection rule, and the threshold-retention rule are recorded before any fit. The single-worker choice trades speed for stable, non-oversubscribed Windows execution. No result has influenced a metric or decision rule, so model construction is allowed.

## 14. Define a compact, evidence-directed model comparison

**Objective.** Correct the source's zero-model gap with the smallest useful stack and limited hyperparameter contrasts.

**Previous state.** There are 26,049 development rows, about 90 transformed columns, moderate imbalance, five shared folds, and a dense matrix below 16 MiB per fold.

**Dataset alignment and logic.** A dummy measures no-skill behavior; regularized logistic regression tests a linear boundary; a random forest tests bagging/nonlinearity; histogram boosting is documented as efficient for `n >= 10,000`; and a compact early-stopped MLP tests a neural baseline. Larger searches are rejected because 55 fold fits already provide comparative evidence.

| Family | Installed 1.8 defaults and tested starting point | Direction, interaction, and limited contrast |
|---|---|---|
| Dummy | `strategy='prior'` is the explicit no-skill start. | It learns only class prevalence; there is no useful complexity search. |
| Logistic | Defaults include `C=1`, no class weight, `lbfgs`, and `max_iter=100`; the start raises only the convergence ceiling to 1,000. | Smaller `C` means stronger shrinkage. Balanced weights increase minority mistakes' influence, usually raising minority recall while lowering precision/accuracy. `C={0.1,1}` × weights tests that interaction; iterations are checked, not assumed. |
| Random forest | The start uses the 1.8 defaults `100` trees, leaf size `1`, `sqrt` features, and no class weight; seed and one worker are explicit. | More trees mainly reduce ensemble variance but cost more. Larger leaves smooth individual trees and can reduce overfit; fewer candidate features increase tree diversity. Leaf sizes `2/5` are paired with balanced bootstrap weights, so this is an interaction contrast—not an isolated causal effect. |
| Histogram boosting | Defaults include learning rate `0.1`, 100 iterations, 31 leaves, leaf size 20, L2 `0`, and no class weight. The start allows 200 iterations and forces early stopping. | Lower learning rate normally needs more iterations. Fewer leaves, larger leaves, and higher L2 reduce capacity together. The balanced candidate isolates class weight at the starting complexity; the regularized candidate jointly tests `0.05/300/15/30/L2=1`. |
| MLP | Defaults include hidden width 100, `alpha=0.0001`, auto batch size, and no early stopping. The measured dense input motivates a smaller width 32, batch 256, and internal early stopping. | More width raises capacity and cost; larger `alpha` increases L2 shrinkage; patience/max iterations interact with early stopping. Only one MLP start is run: further tuning is allowed only if its train/validation evidence is competitive. |

Official parameter definitions: [logistic](https://scikit-learn.org/1.8/modules/generated/sklearn.linear_model.LogisticRegression.html), [forest](https://scikit-learn.org/1.8/modules/generated/sklearn.ensemble.RandomForestClassifier.html), [histogram boosting](https://scikit-learn.org/1.8/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html), [MLP](https://scikit-learn.org/1.8/modules/generated/sklearn.neural_network.MLPClassifier.html).

**New state / success.** Create 11 deterministic, unfitted pipelines and a readable configuration table. These are candidates, not claimed winners.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

candidate_models = {
    "Dummy_prior": DummyClassifier(strategy="prior"),
    "Logistic_start": LogisticRegression(solver="lbfgs", C=1.0, max_iter=1000),
    "Logistic_balanced_C0.1": LogisticRegression(
        solver="lbfgs", C=0.1, class_weight="balanced", max_iter=1000),
    "Logistic_balanced_C1": LogisticRegression(
        solver="lbfgs", C=1.0, class_weight="balanced", max_iter=1000),
    "RandomForest_start": RandomForestClassifier(
        n_estimators=100, min_samples_leaf=1, max_features="sqrt",
        class_weight=None, n_jobs=1, random_state=RANDOM_STATE),
    "RandomForest_balanced_leaf2": RandomForestClassifier(
        n_estimators=100, min_samples_leaf=2, max_features="sqrt",
        class_weight="balanced_subsample", n_jobs=1, random_state=RANDOM_STATE),
    "RandomForest_balanced_leaf5": RandomForestClassifier(
        n_estimators=100, min_samples_leaf=5, max_features="sqrt",
        class_weight="balanced_subsample", n_jobs=1, random_state=RANDOM_STATE),
    "HistGB_start": HistGradientBoostingClassifier(
        learning_rate=0.1, max_iter=200, max_leaf_nodes=31, min_samples_leaf=20,
        early_stopping=True, random_state=RANDOM_STATE),
    "HistGB_balanced": HistGradientBoostingClassifier(
        learning_rate=0.1, max_iter=200, max_leaf_nodes=31, min_samples_leaf=20,
        early_stopping=True, class_weight="balanced", random_state=RANDOM_STATE),
    "HistGB_balanced_regularized": HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=300, max_leaf_nodes=15, min_samples_leaf=30,
        l2_regularization=1.0, early_stopping=True, class_weight="balanced",
        random_state=RANDOM_STATE),
    "MLP_start": MLPClassifier(
        hidden_layer_sizes=(32,), alpha=0.0001, batch_size=256, max_iter=200,
        early_stopping=True, validation_fraction=0.15, n_iter_no_change=15,
        random_state=RANDOM_STATE),
}
candidate_pipelines = {
    name: Pipeline([("preprocess", clone(preprocessor)), ("model", model)])
    for name, model in candidate_models.items()
}
candidate_table = pd.DataFrame({
    "family": ["dummy", "logistic", "logistic", "logistic", "forest", "forest",
               "forest", "histogram boosting", "histogram boosting",
               "histogram boosting", "MLP"],
    "role": ["baseline", "starting", "class/C contrast", "class/C contrast",
             "starting", "class/leaf contrast", "class/leaf contrast", "starting",
             "class contrast", "complexity contrast", "starting neural baseline"],
}, index=candidate_models)

assert len(candidate_pipelines) == 11
assert all(not hasattr(pipe.named_steps["preprocess"], "transformers_")
           for pipe in candidate_pipelines.values())
candidate_table

**Verification.** Eleven unfitted configurations span exactly the five model families promised by the source. The named rows are documented starting values—not mislabeled defaults—and altered rows state whether they test class weighting or a joint complexity interaction. `max_iter` is a ceiling, early stopping controls actual boosting/MLP iterations, and all stochastic estimators use seed 42. No broad search object or extra model family was introduced; shared-fold evaluation is allowed.

## 15. Run the shared development cross-validation benchmark

**Objective.** Compare candidates under identical data boundaries, metrics, and preprocessing while measuring fit gaps, fold variation, convergence, and cost.

**Previous state.** `candidate_pipelines`, `scoring`, and the materialized `cv_splits` are verified and unfitted.

**Dataset alignment and logic.** Training and validation balanced accuracy diagnose under/overfit tendencies; five-fold standard deviation measures split sensitivity; timing measures complexity. Any warning is actionable, so it is captured and fails the cell rather than being suppressed.

**New state / success.** Create `benchmark_results` and `convergence_results`. Success requires 11 complete rows, finite primary scores, zero warnings, and the template preprocessors remaining unfitted.

In [ ]:
from sklearn.model_selection import cross_validate

benchmark_rows = []
convergence_rows = []
benchmark_warnings = []
for name, pipeline in candidate_pipelines.items():
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        result = cross_validate(
            pipeline, X_development, y_development, cv=cv_splits,
            scoring=scoring, return_train_score=True, return_estimator=True,
            n_jobs=1, error_score="raise",
        )
    benchmark_warnings.extend(f"{name}: {item.message}" for item in captured)
    iterations = []
    for estimator in result.pop("estimator"):
        value = getattr(estimator.named_steps["model"], "n_iter_", None)
        if value is not None:
            iterations.append(int(np.max(np.asarray(value))))
    benchmark_rows.append({
        "configuration": name,
        "train_balanced_accuracy": result["train_balanced_accuracy"].mean(),
        "validation_balanced_accuracy": result["test_balanced_accuracy"].mean(),
        "validation_balanced_accuracy_std": result["test_balanced_accuracy"].std(ddof=1),
        "validation_average_precision": result["test_average_precision"].mean(),
        "validation_roc_auc": result["test_roc_auc"].mean(),
        "validation_accuracy": result["test_accuracy"].mean(),
        "validation_precision": result["test_precision"].mean(),
        "validation_recall": result["test_recall"].mean(),
        "validation_f1": result["test_f1"].mean(),
        "mean_fit_seconds": result["fit_time"].mean(),
        "mean_score_seconds": result["score_time"].mean(),
    })
    convergence_rows.append({
        "configuration": name,
        "mean_iterations": np.mean(iterations) if iterations else np.nan,
        "maximum_iterations": max(iterations) if iterations else np.nan,
    })
benchmark_results = pd.DataFrame(benchmark_rows).set_index("configuration")
benchmark_results["train_validation_gap"] = (
    benchmark_results["train_balanced_accuracy"]
    - benchmark_results["validation_balanced_accuracy"]
)
convergence_results = pd.DataFrame(convergence_rows).set_index("configuration")

assert benchmark_warnings == [], "Warnings require diagnosis:\n" + "\n".join(benchmark_warnings)
assert len(benchmark_results) == len(candidate_pipelines)
assert np.isfinite(benchmark_results["validation_balanced_accuracy"]).all()
assert all(not hasattr(pipe.named_steps["preprocess"], "transformers_")
           for pipe in candidate_pipelines.values())
display(benchmark_results.sort_values(
    "validation_balanced_accuracy", ascending=False
).round(4))
display(convergence_results.dropna(how="all").round(1))

**Verification.** All 55 fold fits completed with no warning or failed score. Balanced histogram boosting led validation (`0.8433 ± 0.0053` regularized; `0.8427 ± 0.0051` faster start). Class weighting raised the unweighted booster's positive recall from `0.6437` to `0.8648`, while precision fell from `0.7744` to `0.6044`; AP/AUC stayed essentially unchanged, so weighting moved the operating-point trade-off rather than improving discrimination. The unregularized forest clearly overfit (`0.9702` train vs `0.7702` validation); leaf size 5 reduced its balanced-weight gap to `0.0223`. Balanced logistic gaps were only `0.0009–0.0022` but validation stayed near `0.814`, evidence of a weaker linear representation rather than proof of quality. The MLP was also noncompetitive (`0.7683` validation), so extra neural tuning is rejected. Iteration maxima stayed below their ceilings, and the template pipelines remain unfitted. Development-only selection is allowed.

## 16. Select one configuration with a predeclared efficiency tie-break

**Objective.** Choose the final model without test information and without overstating a tiny unstable advantage.

**Previous state.** `benchmark_results` contains five-fold means, standard deviations, and measured fit time for all 11 candidates; `test_access_log` remains empty.

**Dataset alignment and logic.** Balanced accuracy is primary. The rule is: find the highest mean, define its one-standard-error floor, then choose the fastest measured configuration above that floor. This prefers computational efficiency when apparent differences are smaller than fold uncertainty.

**New state / success.** Create `selected_configuration`, `selected_pipeline`, and `selection_summary`. Success requires selection from development columns only and no test access.

In [ ]:
ranked_results = benchmark_results.sort_values(PRIMARY_RESULT_COLUMN, ascending=False)
top_configuration = ranked_results.index[0]
top_mean = ranked_results.iloc[0][PRIMARY_RESULT_COLUMN]
top_standard_error = (
    ranked_results.iloc[0]["validation_balanced_accuracy_std"] / np.sqrt(len(cv_splits))
)
selection_floor = top_mean - top_standard_error
eligible_results = benchmark_results[
    benchmark_results[PRIMARY_RESULT_COLUMN] >= selection_floor
].copy()
selected_configuration = eligible_results.sort_values(
    ["mean_fit_seconds", PRIMARY_RESULT_COLUMN], ascending=[True, False]
).index[0]
selected_pipeline = candidate_pipelines[selected_configuration]
all_selected_parameters = selected_pipeline.named_steps["model"].get_params()
relevant_parameter_names = [
    name for name in [
        "C", "class_weight", "n_estimators", "max_features", "learning_rate",
        "max_iter", "max_leaf_nodes", "min_samples_leaf", "l2_regularization",
        "hidden_layer_sizes", "alpha", "batch_size", "early_stopping",
        "validation_fraction", "n_iter_no_change", "random_state",
    ] if name in all_selected_parameters
]
selected_hyperparameters = pd.Series({
    name: all_selected_parameters[name] for name in relevant_parameter_names
}, name="validated_final_value")
selection_scope = "development_only"
selection_summary = pd.Series({
    "highest-mean configuration": top_configuration,
    "highest mean balanced accuracy": top_mean,
    "one-standard-error floor": selection_floor,
    "eligible configurations": len(eligible_results),
    "selected configuration": selected_configuration,
    "selection data scope": selection_scope,
}, name="selection_value")

assert selected_configuration in eligible_results.index
assert selection_scope == "development_only" and test_access_log == []
display(eligible_results.sort_values("mean_fit_seconds").round(4))
display(selection_summary.to_frame())
display(selected_hyperparameters.to_frame())

**Verification.** `HistGB_balanced_regularized` had the highest mean (`0.843263`), but both balanced boosters exceeded the `0.840913` one-standard-error floor. `HistGB_balanced` was selected because its mean fold fit was `0.5684 s` versus `1.1067 s`, while giving up only `0.000608` balanced accuracy. The displayed final values are learning rate `0.1`, `max_iter=200`, 31 leaves, minimum leaf 20, L2 `0`, early stopping, balanced class weights, and seed 42. This efficiency heuristic uses correlated folds and one timing run; it is a validated choice, not proof of superiority. The locked test still has no prediction or metric, so OOF threshold assessment is allowed.

## 17. Generate out-of-fold probabilities and test the decision threshold

**Objective.** Examine whether the selected probabilistic model's default 0.5 decision rule treats both classes adequately.

**Previous state.** `selected_pipeline` was chosen only from shared development CV; `cv_splits` covers each row once; the test access log is empty.

**Dataset alignment and logic.** The 24.08% positive rate makes threshold behavior important. Probabilities are generated out-of-fold, never in-sample. Thresholds 0.10–0.90 are compared by mean fold balanced accuracy. The default changes only if the top gain exceeds its fold standard error, limiting threshold overfit. Scikit-learn separates probability estimation from decisions in its [threshold guide](https://scikit-learn.org/1.8/modules/classification_threshold.html).

**New state / success.** Create `oof_probability`, `threshold_results`, and `decision_threshold`. Success requires one probability per development row, no warnings, and no test access.

In [ ]:
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import cross_val_predict

with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter("always")
    oof_probability = cross_val_predict(
        selected_pipeline, X_development, y_development, cv=cv_splits,
        method="predict_proba", n_jobs=1,
    )[:, 1]
threshold_rows = []
for threshold in np.round(np.arange(0.10, 0.901, 0.01), 2):
    fold_scores = [
        balanced_accuracy_score(
            y_development.iloc[valid_position],
            oof_probability[valid_position] >= threshold,
        )
        for _, valid_position in cv_splits
    ]
    threshold_rows.append({
        "threshold": threshold, "mean_balanced_accuracy": np.mean(fold_scores),
        "std_balanced_accuracy": np.std(fold_scores, ddof=1),
    })
threshold_results = pd.DataFrame(threshold_rows).set_index("threshold")
top_threshold = threshold_results["mean_balanced_accuracy"].idxmax()
top_threshold_row = threshold_results.loc[top_threshold]
default_threshold_row = threshold_results.loc[0.50]
threshold_gain = (
    top_threshold_row["mean_balanced_accuracy"]
    - default_threshold_row["mean_balanced_accuracy"]
)
threshold_standard_error = top_threshold_row["std_balanced_accuracy"] / np.sqrt(len(cv_splits))
decision_threshold = float(top_threshold if threshold_gain > threshold_standard_error else 0.50)

assert captured == [] and len(oof_probability) == len(y_development)
assert np.isfinite(oof_probability).all() and np.all((oof_probability >= 0) & (oof_probability <= 1))
assert test_access_log == []
display(pd.DataFrame({
    "default_0.50": default_threshold_row,
    "development_optimum": top_threshold_row,
}).T.round(4))
print(
    f"Top threshold={top_threshold:.2f}; gain={threshold_gain:.4f}; "
    f"top SE={threshold_standard_error:.4f}; selected={decision_threshold:.2f}"
)

**Verification.** Every development row received exactly one finite out-of-fold probability with no warning. Threshold `0.47` reached `0.8436 ± 0.0058`, versus `0.8427 ± 0.0051` at `0.50`; its `0.0009` gain was below the `0.0026` standard error, so the predeclared rule retained `0.50`. The test remains untouched; class-level OOF diagnosis is allowed.

## 18. Diagnose development class behavior and probability quality

**Objective.** Measure per-class errors, ranking, and probability quality before final fitting.

**Previous state.** `oof_probability` is purely out-of-fold and `decision_threshold` was chosen without test data.

**Dataset alignment and logic.** Overall accuracy can hide the positive class. A confusion matrix, per-class precision/recall/F1/support, balanced accuracy, ROC AUC, average precision, and Brier loss expose distinct behavior. Similar train and validation scores alone do not prove quality.

**New state / success.** Create `oof_prediction`, `oof_metrics`, `oof_confusion`, and `oof_class_report`. Success requires all 26,049 rows to be accounted for.

In [ ]:
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    classification_report, confusion_matrix, f1_score,
    recall_score, roc_auc_score,
)

oof_prediction = (oof_probability >= decision_threshold).astype("int8")
oof_confusion = pd.DataFrame(
    confusion_matrix(y_development, oof_prediction),
    index=["actual_<=50K", "actual_>50K"],
    columns=["predicted_<=50K", "predicted_>50K"],
)
oof_class_report = pd.DataFrame(classification_report(
    y_development, oof_prediction, target_names=["<=50K", ">50K"],
    output_dict=True, zero_division=0,
)).T.drop(index="accuracy")
oof_metrics = pd.Series({
    "accuracy": accuracy_score(y_development, oof_prediction),
    "balanced_accuracy": balanced_accuracy_score(y_development, oof_prediction),
    "positive_precision": precision_score(y_development, oof_prediction, zero_division=0),
    "positive_recall": recall_score(y_development, oof_prediction),
    "positive_f1": f1_score(y_development, oof_prediction),
    "roc_auc": roc_auc_score(y_development, oof_probability),
    "average_precision": average_precision_score(y_development, oof_probability),
    "brier_loss": brier_score_loss(y_development, oof_probability),
}, name="OOF_value")

assert oof_confusion.to_numpy().sum() == len(y_development)
assert oof_class_report.loc[["<=50K", ">50K"], "support"].sum() == len(y_development)
display(oof_metrics.to_frame().round(4))
display(oof_confusion)
display(oof_class_report.round(4))

**Verification.** All 26,049 OOF rows are accounted for. Balanced accuracy is `0.8427`: positive recall is `0.8648` and negative recall `0.8205`, but positive precision is only `0.6045` because 3,550 negative rows are false positives. ROC AUC is `0.9255`, AP `0.8212`, and Brier loss `0.1111`. This is strong ranking and recall under the declared metric, not proof that the false-positive trade-off is useful. These remain validation estimates; feature-policy sensitivity must be checked before final fitting.

## 19. Declare feature-policy ablations

**Objective.** Test four source-promised representation risks without changing the selected model family or validation boundaries.

**Previous state.** The selected configuration and OOF behavior are development-only; measured evidence flagged capital timing, sensitive attributes, redundant education representation, and `fnlwgt` semantics.

**Dataset alignment and logic.** Each ablation changes one policy relative to the 12-feature primary input: remove capital, remove `race`/`sex`, replace ordinal education with one-hot text education, or add `fnlwgt` as a conventional predictor. They answer sensitivity questions, not causal ones.

**New state / success.** Create `ablation_specs` and a feature-count table. Success requires each feature set to exist in development data and exclude target columns.

In [ ]:
ablation_specs = {
    "primary": {
        "regular": regular_numeric, "capital": capital_numeric,
        "categorical": categorical_features,
    },
    "no_capital": {
        "regular": regular_numeric, "capital": [], "categorical": categorical_features,
    },
    "no_sensitive": {
        "regular": regular_numeric, "capital": capital_numeric,
        "categorical": [c for c in categorical_features if c not in {"race", "sex"}],
    },
    "education_onehot": {
        "regular": ["age", "hours_per_week"], "capital": capital_numeric,
        "categorical": categorical_features + ["education"],
    },
    "fnlwgt_predictor": {
        "regular": regular_numeric + ["fnlwgt"], "capital": capital_numeric,
        "categorical": categorical_features,
    },
}
ablation_feature_table = pd.DataFrame({
    name: {
        "regular_numeric": len(spec["regular"]), "capital_numeric": len(spec["capital"]),
        "categorical": len(spec["categorical"]),
        "total_raw_features": sum(len(spec[key]) for key in ("regular", "capital", "categorical")),
    }
    for name, spec in ablation_specs.items()
}).T

assert all(set(sum((spec[key] for key in ("regular", "capital", "categorical")), [])).issubset(development.columns)
           for spec in ablation_specs.values())
assert all({"target", "income"}.isdisjoint(sum(
    (spec[key] for key in ("regular", "capital", "categorical")), []
)) for spec in ablation_specs.values())
ablation_feature_table

**Verification.** Five explicit policies are defined: the primary 12 features plus four one-change sensitivities containing 10–13 raw features. Every named field exists and target columns are absent. The selected estimator and shared folds remain unchanged, so comparative ablation CV is allowed.

## 20. Execute the feature-policy ablations

**Objective.** Quantify how each policy changes validation performance and generalization gap.

**Previous state.** `ablation_specs` passed schema checks; `selected_pipeline`, `cv_splits`, and primary benchmark results are fixed.

**Dataset alignment and logic.** Reusing the selected model and exact folds isolates representation effects. The already-computed primary row is reused instead of spending five redundant fits. Each alternative rebuilds preprocessing from its declared roles and remains fold-local.

**New state / success.** Create `ablation_results`. Success requires five complete policies, zero warnings, finite metrics, and an empty test access log.

In [ ]:
selected_row = benchmark_results.loc[selected_configuration]
ablation_rows = [{
    "policy": "primary",
    "train_balanced_accuracy": selected_row["train_balanced_accuracy"],
    "validation_balanced_accuracy": selected_row["validation_balanced_accuracy"],
    "validation_balanced_accuracy_std": selected_row["validation_balanced_accuracy_std"],
    "validation_average_precision": selected_row["validation_average_precision"],
    "validation_roc_auc": selected_row["validation_roc_auc"],
}]
ablation_warnings = []
for policy, spec in list(ablation_specs.items())[1:]:
    transformers = [("numeric", clone(regular_numeric_pipeline), spec["regular"])]
    if spec["capital"]:
        transformers.append(("capital", clone(capital_pipeline), spec["capital"]))
    transformers.append(("categorical", clone(categorical_pipeline), spec["categorical"]))
    variant_preprocessor = ColumnTransformer(transformers, verbose_feature_names_out=False)
    variant_features = spec["regular"] + spec["capital"] + spec["categorical"]
    variant_pipeline = Pipeline([
        ("preprocess", variant_preprocessor),
        ("model", clone(selected_pipeline.named_steps["model"])),
    ])
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        result = cross_validate(
            variant_pipeline, development[variant_features], y_development,
            cv=cv_splits, scoring={
                "balanced_accuracy": "balanced_accuracy",
                "average_precision": "average_precision", "roc_auc": "roc_auc",
            }, return_train_score=True, n_jobs=1, error_score="raise",
        )
    ablation_warnings.extend(f"{policy}: {item.message}" for item in captured)
    ablation_rows.append({
        "policy": policy,
        "train_balanced_accuracy": result["train_balanced_accuracy"].mean(),
        "validation_balanced_accuracy": result["test_balanced_accuracy"].mean(),
        "validation_balanced_accuracy_std": result["test_balanced_accuracy"].std(ddof=1),
        "validation_average_precision": result["test_average_precision"].mean(),
        "validation_roc_auc": result["test_roc_auc"].mean(),
    })
ablation_results = pd.DataFrame(ablation_rows).set_index("policy")
ablation_results["balanced_accuracy_delta_vs_primary"] = (
    ablation_results["validation_balanced_accuracy"]
    - ablation_results.loc["primary", "validation_balanced_accuracy"]
)

assert ablation_warnings == [], "Warnings require diagnosis:\n" + "\n".join(ablation_warnings)
assert len(ablation_results) == len(ablation_specs)
assert np.isfinite(ablation_results.to_numpy()).all() and test_access_log == []
ablation_results.round(4)

**Verification.** All four alternatives completed on the same folds without warnings. Removing capital reduced validation balanced accuracy by `0.0324` and AP from `0.8214` to `0.7123`, confirming material dependence on potentially target-adjacent fields. Removing `race`/`sex` changed balanced accuracy by only `-0.0004`; one-hot education by `-0.0010`; adding `fnlwgt` by `+0.0006` while AP fell. Those small changes are within fold variation and do not justify changing the canonical primary policy. The test is still untouched, so final development fitting is allowed.

## 21. Fit the selected pipeline on all development rows

**Objective.** Train exactly one final pipeline after all model, hyperparameter, threshold, and feature-policy decisions are frozen.

**Previous state.** `selected_configuration`, `decision_threshold`, and primary feature policy were chosen from development-only evidence; all ablations are complete; `test_access_log` is empty.

**Dataset alignment and logic.** Fitting the entire pipeline on 26,049 development rows lets preprocessing learn only from development and uses all permitted training evidence. Fit time, training balanced accuracy, and actual iteration count provide a final convergence/overfit reference.

**New state / success.** Create fitted `final_model` and `final_fit_summary`. Success requires no warning, a fitted internal preprocessor, and no test access.

In [ ]:
from time import perf_counter

final_model = clone(selected_pipeline)
fit_start = perf_counter()
with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter("always")
    final_model.fit(X_development, y_development)
final_fit_seconds = perf_counter() - fit_start
final_train_prediction = (
    final_model.predict_proba(X_development)[:, 1] >= decision_threshold
).astype("int8")
final_model_iterations = getattr(final_model.named_steps["model"], "n_iter_", np.nan)
final_fit_summary = pd.Series({
    "selected configuration": selected_configuration,
    "development rows fitted": len(X_development),
    "fit seconds": final_fit_seconds,
    "iterations": int(np.max(np.asarray(final_model_iterations)))
    if np.size(final_model_iterations) else np.nan,
    "development resubstitution balanced accuracy": balanced_accuracy_score(
        y_development, final_train_prediction),
    "decision threshold": decision_threshold,
}, name="final_fit_value")

assert captured == [] and hasattr(final_model.named_steps["preprocess"], "transformers_")
assert test_access_log == [] and selection_scope == "development_only"
final_fit_summary.to_frame()

**Verification.** `HistGB_balanced` fitted all 26,049 development rows with no warning and stopped at 100 of the 200 allowed iterations. Resubstitution balanced accuracy is `0.8633`, compared with the earlier `0.8427` validation mean—a modest overfit tendency, not a generalization claim. All choices were frozen and the test access log is still empty, so the single final prediction batch is allowed.

## 22. Create the one allowed locked-test prediction batch

**Objective.** Evaluate the frozen pipeline and threshold on unseen rows exactly once.

**Previous state.** `final_model` is fitted only on development data; the selected configuration, feature policy, and threshold are frozen; `test_access_log == []`.

**Dataset alignment and logic.** `X_locked_test` has the same 12 columns/order as training and group/index overlap is zero. A test-target alias and all target-based test calculations are created only now, after selection, thresholding, and ablations; the reserved `locked_test` frame itself necessarily retained its source columns.

**New state / success.** Create `y_locked_test`, `test_probability`, and `test_prediction`, then record one access event. Success requires aligned indexes, finite probabilities, and exactly one prediction batch.

In [ ]:
assert test_access_log == [], "The locked test was already predicted."
y_locked_test = locked_test["target"].copy()
survey_weight_locked_test = locked_test["fnlwgt"].copy()
test_probability = final_model.predict_proba(X_locked_test)[:, 1]
test_prediction = (test_probability >= decision_threshold).astype("int8")
test_access_log.append("final_prediction_batch")

assert X_locked_test.index.equals(y_locked_test.index)
assert len(test_probability) == len(y_locked_test) == 6_512
assert np.isfinite(test_probability).all()
assert len(test_access_log) == 1
pd.Series({
    "test rows predicted": len(test_prediction),
    "actual positive rate pct": 100 * y_locked_test.mean(),
    "predicted positive rate pct": 100 * test_prediction.mean(),
    "prediction batches": len(test_access_log),
}, name="final_test_state").to_frame()

**Verification.** The output confirms 6,512 aligned rows, finite scores, and exactly one prediction batch after all decisions were frozen. Actual positives are `24.08%`, while predicted positives are `34.46%`, already signaling the balanced-weight recall/false-positive trade-off. Model or threshold reselection is no longer allowed; only diagnostic reporting may follow.

## 23. Report final overall and per-class performance

**Objective.** Quantify final generalization, confusion patterns, class treatment, ranking, and probability quality.

**Previous state.** `y_locked_test`, `test_probability`, and `test_prediction` came from the single frozen test prediction batch.

**Dataset alignment and logic.** Moderate imbalance requires negative recall (specificity), positive recall, precision, F1, balanced accuracy, and support in addition to accuracy. ROC AUC/AP evaluate ranking; Brier loss evaluates probability error but does not prove calibration.

**New state / success.** Create `test_metrics`, `test_confusion`, and `test_class_report`. Success requires all 6,512 rows to be accounted for and no additional model call.

In [ ]:
test_confusion_values = confusion_matrix(y_locked_test, test_prediction)
true_negative, false_positive = test_confusion_values[0]
test_confusion = pd.DataFrame(
    test_confusion_values,
    index=["actual_<=50K", "actual_>50K"],
    columns=["predicted_<=50K", "predicted_>50K"],
)
test_class_report = pd.DataFrame(classification_report(
    y_locked_test, test_prediction, target_names=["<=50K", ">50K"],
    output_dict=True, zero_division=0,
)).T.drop(index="accuracy")
test_metrics = pd.Series({
    "accuracy": accuracy_score(y_locked_test, test_prediction),
    "balanced_accuracy": balanced_accuracy_score(y_locked_test, test_prediction),
    "negative_recall_specificity": true_negative / (true_negative + false_positive),
    "positive_precision": precision_score(y_locked_test, test_prediction, zero_division=0),
    "positive_recall": recall_score(y_locked_test, test_prediction),
    "positive_f1": f1_score(y_locked_test, test_prediction),
    "roc_auc": roc_auc_score(y_locked_test, test_probability),
    "average_precision": average_precision_score(y_locked_test, test_probability),
    "brier_loss": brier_score_loss(y_locked_test, test_probability),
}, name="test_value")

assert test_confusion.to_numpy().sum() == len(y_locked_test)
assert test_class_report.loc[["<=50K", ">50K"], "support"].sum() == len(y_locked_test)
assert len(test_access_log) == 1
display(test_metrics.to_frame().round(4))
display(test_confusion)
display(test_class_report.round(4))

**Verification.** All 6,512 test rows are accounted for. Balanced accuracy is `0.8472`, specificity `0.8226`, positive recall `0.8718`, precision `0.6092`, and F1 `0.7172`; the confusion matrix contains 877 false positives and 201 false negatives. ROC AUC is `0.9324`, AP `0.8399`, and Brier loss `0.1077`. Test balanced accuracy is within one CV standard deviation of the `0.8427 ± 0.0051` validation result, so it is plausible rather than suspicious. No second prediction batch occurred.

## 24. Inspect probability calibration

**Objective.** Check whether class-weighted scores can be interpreted as original-population income probabilities.

**Previous state.** The frozen test batch contains 6,512 finite probabilities, Brier loss `0.1077`, actual prevalence `24.08%`, and a much higher `34.46%` predicted-positive rate at threshold 0.50.

**Dataset alignment and logic.** Balanced class weights change the fitted loss contribution of each class. Ten equal-frequency bins provide about 651 rows per bin and directly compare mean predicted score with observed positive fraction. This is a final diagnostic only; it cannot trigger calibration or reselection on the test set.

**New state / success.** Create `calibration_table` and a support-weighted absolute bin gap. Success requires ten nonempty bins covering all test rows and no additional prediction call.

In [ ]:
calibration_frame = pd.DataFrame({
    "actual": y_locked_test.to_numpy(), "probability": test_probability,
})
calibration_frame["probability_bin"] = pd.qcut(
    calibration_frame["probability"], q=10, duplicates="drop"
)
calibration_table = calibration_frame.groupby(
    "probability_bin", observed=True
).agg(
    support=("actual", "size"),
    mean_predicted_probability=("probability", "mean"),
    observed_positive_rate=("actual", "mean"),
)
calibration_table["absolute_gap"] = (
    calibration_table["mean_predicted_probability"]
    - calibration_table["observed_positive_rate"]
).abs()
quantile_binned_absolute_gap = np.average(
    calibration_table["absolute_gap"], weights=calibration_table["support"]
)

assert len(calibration_table) == 10
assert calibration_table["support"].sum() == len(y_locked_test)
assert len(test_access_log) == 1
display(calibration_table.round(4))
print(f"Support-weighted absolute 10-bin gap: {quantile_binned_absolute_gap:.4f}")

**Verification.** Ten nonempty equal-frequency bins cover all 6,512 rows. The support-weighted absolute bin gap is `0.1049`, with especially large mid/high-score overestimation. This class-weighted model's scores are useful for ranking and the declared decision rule, but they must not be read as calibrated income probabilities. Calibration for real decisions would need a development-only design and another untouched evaluation set; this test output cannot be used to refit.

## 25. Separate survey-weighted evaluation from predictor use

**Objective.** Show how CPS final weights change aggregate test metrics without treating `fnlwgt` as a personal predictor.

**Previous state.** Frozen test predictions and `survey_weight_locked_test` are aligned; the `fnlwgt_predictor` development ablation separately tested feature use.

**Dataset alignment and logic.** UCI documents `fnlwgt` as a CPS population weight. Sample-weighted metrics estimate a different population-weighted quantity; they must not be conflated with adding the weight as a feature.

**New state / success.** Create `weight_sensitivity`. Success requires positive finite weights and identical predictions in weighted/unweighted columns.

In [ ]:
weight_sensitivity = pd.DataFrame({
    "unweighted": {
        "accuracy": accuracy_score(y_locked_test, test_prediction),
        "balanced_accuracy": balanced_accuracy_score(y_locked_test, test_prediction),
        "positive_recall": recall_score(y_locked_test, test_prediction),
        "positive_f1": f1_score(y_locked_test, test_prediction),
        "average_precision": average_precision_score(y_locked_test, test_probability),
    },
    "survey_weighted": {
        "accuracy": accuracy_score(
            y_locked_test, test_prediction, sample_weight=survey_weight_locked_test),
        "balanced_accuracy": balanced_accuracy_score(
            y_locked_test, test_prediction, sample_weight=survey_weight_locked_test),
        "positive_recall": recall_score(
            y_locked_test, test_prediction, sample_weight=survey_weight_locked_test),
        "positive_f1": f1_score(
            y_locked_test, test_prediction, sample_weight=survey_weight_locked_test),
        "average_precision": average_precision_score(
            y_locked_test, test_probability, sample_weight=survey_weight_locked_test),
    },
})

assert survey_weight_locked_test.index.equals(y_locked_test.index)
assert np.isfinite(survey_weight_locked_test).all() and survey_weight_locked_test.gt(0).all()
assert len(test_access_log) == 1
weight_sensitivity.round(4)

**Verification.** The same predictions yield unweighted versus survey-weighted balanced accuracy `0.8472` versus `0.8494`; recall `0.8718` versus `0.8669`; and AP `0.8399` versus `0.8371`. These small changes are population-weight sensitivity, not a new model. Positive finite weights and indexes align, and the prediction batch count remains one.

## 26. Audit final errors by sex and race

**Objective.** Measure whether error rates differ across observed demographic groups without claiming fairness or causality.

**Previous state.** Final predictions are frozen, and development evidence showed large differences in group sizes and historical label rates.

**Dataset alignment and logic.** Each slice reports support, positive/negative denominators, true-positive rate, false-positive rate, precision, and accuracy. Denominators make small-group uncertainty visible. Metric parity alone cannot establish fairness; see [NIST AI RMF](https://airc.nist.gov/airmf-resources/airmf/5-sec-core/) and [Fairlearn assessment guidance](https://fairlearn.org/main/user_guide/assessment/perform_fairness_assessment.html).

**New state / success.** Create `sex_error_audit` and `race_error_audit`. Success requires each table's support to sum to all 6,512 test rows and no new prediction call.

In [ ]:
subgroup_tables = {}
for attribute in ["sex", "race"]:
    subgroup_rows = []
    for group_value, positions in locked_test.groupby(attribute, sort=True).indices.items():
        group_y = y_locked_test.iloc[positions].to_numpy()
        group_prediction = test_prediction[positions]
        group_true_negative, group_false_positive, group_false_negative, group_true_positive = (
            confusion_matrix(group_y, group_prediction, labels=[0, 1]).ravel()
        )
        predicted_positive = group_true_positive + group_false_positive
        actual_positive = group_true_positive + group_false_negative
        actual_negative = group_true_negative + group_false_positive
        assert actual_positive > 0 and actual_negative > 0 and predicted_positive > 0, (
            f"Undefined {attribute} metric for group {group_value!r}; report it explicitly."
        )
        subgroup_rows.append({
            attribute: group_value, "support": len(positions),
            "positives": int(group_y.sum()), "negatives": int((group_y == 0).sum()),
            "true_positive_rate": group_true_positive / actual_positive,
            "false_positive_rate": group_false_positive / actual_negative,
            "precision": group_true_positive / predicted_positive,
            "accuracy": (group_true_positive + group_true_negative) / len(positions),
        })
    subgroup_tables[attribute] = pd.DataFrame(subgroup_rows).set_index(attribute)
sex_error_audit = subgroup_tables["sex"]
race_error_audit = subgroup_tables["race"]

assert sex_error_audit["support"].sum() == len(y_locked_test)
assert race_error_audit["support"].sum() == len(y_locked_test)
assert len(test_access_log) == 1
display(sex_error_audit.round(4))
display(race_error_audit.round(4))

**Verification.** Both tables sum to all 6,512 rows and all denominators are explicitly positive. Female versus male TPR is `0.7723` versus `0.8884`, while FPR is `0.0661` versus `0.2473`. Among better-supported race groups, TPR ranges from `0.7403` (Black) to `0.8843` (White); Asian FPR is `0.2133`. Amer-Indian-Eskimo and Other slices contain only 5 and 4 positive cases, so their rates are highly unstable. These are material observed disparities—not proof of causality, fairness, or unfairness under a defined policy.

## 27. Compare training, validation, OOF, and final test evidence

**Objective.** Summarize underfitting, overfitting, fold stability, and final generalization without treating one score as proof.

**Previous state.** The selected configuration has train/validation CV results, OOF predictions, one final test result, convergence information, and measured cost.

**Dataset alignment and logic.** Strong training with weaker validation suggests overfit; jointly weak scores suggest underfit; a small gap alone does not prove quality. Test performance is interpreted against the five-fold mean and standard deviation, while fit time and iterations describe computational cost.

**New state / success.** Create `generalization_summary`. Success requires finite train, validation, OOF, and test values derived from the same primary pipeline and frozen threshold.

In [ ]:
generalization_summary = pd.DataFrame({
    "balanced_accuracy": {
        "CV training mean": selected_row["train_balanced_accuracy"],
        "CV validation mean": selected_row["validation_balanced_accuracy"],
        "OOF development": oof_metrics["balanced_accuracy"],
        "locked test": test_metrics["balanced_accuracy"],
    },
    "context": {
        "CV training mean": "resampled training folds",
        "CV validation mean": f"std={selected_row['validation_balanced_accuracy_std']:.4f}",
        "OOF development": f"threshold={decision_threshold:.2f}",
        "locked test": "one final prediction batch",
    },
})

assert np.isfinite(generalization_summary["balanced_accuracy"]).all()
assert len(test_access_log) == 1
generalization_summary

**Verification.** Selected-model CV training/validation balanced accuracy is `0.8639/0.8427` (gap `0.0213`), so mild overfit tendency is present but not severe. Because early stopping withholds an internal subset, the reported outer-train score is a hybrid of fit and internal-validation rows, not a pure in-sample score. The locked test is `0.8472`, close to validation and within its `0.0051` fold SD. By contrast, the starting forest's `0.1999` gap is clear overfit; logistic's tiny gaps with only about `0.814` validation balanced accuracy indicate limited linear representation, not goodness. OOF and CV reuse the same folds and are not independent confirmations. Agreement supports internal generalization only.

## 28. Final reproducibility and leakage audit

**Objective.** Verify the notebook's end-state contracts and confirm the original notebook was not modified.

**Previous state.** All diagnostics and final reporting are complete; the single test prediction batch is recorded.

**Dataset alignment and logic.** Trust requires unchanged input/source hashes, frozen split membership, complete fold coverage, group separation, aligned feature/target indexes, unfitted reusable templates, a fitted final pipeline, development-only selection, and exactly one test prediction batch.

**New state / success.** Create `final_audit`. Success requires every Boolean check to be `True`; otherwise execution stops.

In [ ]:
final_data_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
final_source_hash = hashlib.sha256(SOURCE_NOTEBOOK_PATH.read_bytes()).hexdigest()
final_test_digest = hashlib.sha256(
    locked_test.index.to_numpy(dtype=np.int64).astype("<i8", copy=False).tobytes()
).hexdigest()
improved_notebook_source = json.loads(IMPROVED_NOTEBOOK_PATH.read_text(encoding="utf-8"))
improved_code = "\n".join(
    "".join(cell.get("source", []))
    for cell in improved_notebook_source["cells"] if cell["cell_type"] == "code"
)
locked_prediction_pattern = "final_model.predict_proba(" + "X_locked_test)"
locked_target_pattern = "y_locked_test = " + 'locked_test["target"].copy()'
static_locked_prediction_calls = improved_code.count(locked_prediction_pattern)
static_locked_target_extractions = improved_code.count(locked_target_pattern)
final_audit = pd.Series({
    "data file unchanged": final_data_hash == data_hash == EXPECTED_DATA_SHA256,
    "source unchanged during fresh run": final_source_hash == source_notebook_hash == EXPECTED_SOURCE_SHA256,
    "scikit-learn version pinned": sklearn.__version__ == "1.8.0",
    "test membership digest unchanged": final_test_digest == test_index_digest,
    "CV membership digest pinned": cv_split_digest == EXPECTED_CV_SPLIT_DIGEST,
    "development/test indexes disjoint": set(development.index).isdisjoint(locked_test.index),
    "development/test profiles disjoint": set(development_group).isdisjoint(test_group),
    "every development row validated once": bool(np.all(validation_hits == 1)),
    "all CV group overlaps zero": bool(fold_audit["group_overlap"].eq(0).all()),
    "development feature-target indexes aligned": X_development.index.equals(y_development.index),
    "locked-test feature-target indexes aligned": X_locked_test.index.equals(y_locked_test.index),
    "template preprocessor remains unfitted": not hasattr(preprocessor, "transformers_"),
    "final preprocessor fitted on development": hasattr(final_model.named_steps["preprocess"], "transformers_"),
    "selection used development only": selection_scope == "development_only",
    "selection rule declared": bool(SELECTION_RULE and THRESHOLD_RULE),
    "static locked-test prediction call count is one": static_locked_prediction_calls == 1,
    "static locked-target extraction count is one": static_locked_target_extractions == 1,
    "test log records one final batch": test_access_log == ["final_prediction_batch"],
    "calibration bins cover test": calibration_table["support"].sum() == len(y_locked_test),
}, name="passed")

assert final_audit.all(), final_audit[~final_audit].to_dict()
final_audit.to_frame()

**Verification.** Every final Boolean is `True`. Data, source-snapshot, test, and CV digests stayed fixed during this fresh run; all folds are complete and group-disjoint; indexes align; reusable preprocessing stayed unfitted while the final clone fitted on development; and decision rules were declared. Static source inspection independently finds exactly one locked-test prediction call and one locked-target extraction, while the runtime log records that final batch. The current source snapshot was unchanged during execution; a separate concurrent worktree edit observed before this run means this does not claim equality with repository `HEAD` or the session's earlier source hash.

## Honest benchmark conclusion

This is now a complete, internally controlled classification benchmark rather than a preprocessing hand-off. The faster one-standard-error choice, `HistGB_balanced`, achieved validation balanced accuracy `0.8427 ± 0.0051` and locked-test balanced accuracy `0.8472`; test specificity/positive recall were `0.8226/0.8718`, with precision `0.6092`, F1 `0.7172`, ROC AUC `0.9324`, and AP `0.8399`. The selected train–validation gap was `0.0213`, while the unregularized forest's `0.1999` gap demonstrated severe overfit. These results support the selected configuration only for this supplied `adult.data` file, version-pinned internal group-disjoint split, stated feature policy, and retained threshold `0.50`.

The benchmark is **trustworthy within those tested limits** because all learned preprocessing is fold-local, every candidate shares the same validation rows and metrics, selection and threshold work use development data only, and static inspection plus the runtime audit confirm one final locked-test prediction call. Class treatment is explicit but not cost-optimal by proof: the model predicts `34.46%` positive against `24.08%` observed, prioritizing recall and producing 877 false positives. It is **not evidence of demographic fairness**: female/male TPR and FPR differ materially, race slices differ and some have only 4–5 positives, protected-field removal leaves proxies, and no fairness objective or decision context was supplied.

Remaining uncertainty is material. The data describe a selected 1994 Census sample; `$50K` is not inflation-adjusted; only `adult.data` is local, so results are not the official UCI test benchmark; no timestamp, household/person ID, deployment population, or error costs exist; one exact development profile has conflicting labels; and removing capital reduced validation balanced accuracy by `0.0324`, confirming dependence on fields that may be unavailable or target-adjacent prospectively. CPS weights change the estimand; five correlated folds and one timing run do not prove statistical superiority; a single internal holdout does not establish temporal stability; and the compact search does not exhaust all models or hyperparameters. Balanced class weights alter the fitted loss, and the support-weighted ten-bin calibration gap is `0.1049`, so scores must not be interpreted as calibrated income probabilities without a development-only calibration design and another untouched evaluation set. A concurrent edit changed the source worktree before final execution; this run pins that snapshot and proves it stayed unchanged during execution, not that it equals repository `HEAD`. This notebook does not justify hiring, credit, benefits, surveillance, or other high-impact use.